# Notebook 1: Hidden Representation Dimension, Stability, and Sampling

This notebook is the clean replacement for the original representation notebook. It is intentionally split into two stages:

1. **Experiment stage**: call the JAX training/analysis script, which saves logs, checkpoints, representation metrics, stability metrics, samples, and metadata.
2. **Visualization stage**: read saved files and make independent, paper-friendly plots.

The default configuration below uses `D=512`, `AdamW`, a 5-block residual FCN with pre-norm AdaLN-zero, zero-initialized output projection, and dynamic diffusion-style batch sampling.

Set `SAVE_PDF=True` only after the plot style is final. PNG is always saved.

## Metrics

For each hidden representation matrix `H` with rows as samples and columns as hidden features, this notebook reports:

- `stable_rank(H) = ||H||_F^2 / ||H||_op^2`, after centering rows for representation matrices.
- `rank95(H)`, the number of principal components required to explain 95% of centered representation energy.
- The hidden representation spectrum, plotted as normalized singular values `s_i / s_1` on a log scale.
- Noise-resampling stability: for fixed clean samples and fixed `t`, we resample corruption noise and report `Tr Var_j H(x_i, eps_ij) / E ||H(x_i, eps_ij)||^2`, averaged over clean samples.
- Sample quality: Chamfer is symmetric mean nearest-neighbor distance. Precision is the fraction of generated samples within the data manifold radius. Recall is the fraction of data samples covered by generated samples. The radius is the 95th percentile of data-to-data nearest-neighbor distance. Sample legends report the ambient high-dimensional version of these metrics, while the 2D panel keeps the original ground-truth projection window.


In [ ]:
from pathlib import Path
import os
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

import subprocess
import pandas as pd
from IPython.display import display, Markdown

from clean_jax_exp.visualize import (
    latest_run,
    plot_train_loss,
    plot_representation_bar,
    plot_stability_bar,
    plot_samples,
    plot_samples_pca3d,
    plot_sample_metric,
    plot_representation_spectrum,
)

RUN_TRAINING = False
SAVE_PDF = False
OUTPUT_ROOT = Path("results/clean_jax_representation")

TRAIN_CMD = [
    "/Users/tongtongliang/miniforge3/bin/python3.12",
    "run_clean_jax_experiment.py",
    "--output-root", str(OUTPUT_ROOT),
    "--ambient-dim", "512",
    "--n-samples", "8192",
    "--width", "256",
    "--depth", "5",
    "--time-embed-dim", "256",
    "--steps", "100000",
    "--batch-size", "256",
        "--loss-every", "100",
    "--print-every", "1000",
    "--lr", "1e-4",
    "--grad-clip-norm", "1.0",
    "--save-checkpoints",
]

if RUN_TRAINING:
    subprocess.run(TRAIN_CMD, check=True)

RUN_DIR = latest_run(OUTPUT_ROOT / "runs")
print(f"Using run: {RUN_DIR}")

In [ ]:
metadata = pd.read_json(RUN_DIR / "metadata.json", typ="series")
display(metadata)

loss_df = pd.read_csv(RUN_DIR / "logs" / "loss.csv")
repr_df = pd.read_csv(RUN_DIR / "analysis" / "representation_metrics.csv")
stability_df = pd.read_csv(RUN_DIR / "analysis" / "representation_stability.csv")
sample_metrics_df = pd.read_csv(RUN_DIR / "analysis" / "sample_metrics.csv")

print("loss rows", len(loss_df))
print("representation rows", len(repr_df))
print("stability rows", len(stability_df))
display(sample_metrics_df)

## Training Loss

The loss is the unified velocity loss for all output parameterizations. The backbone output differs (`x`, `v`, `eps`), but every model is optimized through the same velocity-space objective.

In [ ]:
plot_train_loss(RUN_DIR, save_pdf=SAVE_PDF)

## Mixed-time Representation Dimension

Mixed time means each sample receives an independently sampled `t`, matching the training distribution more closely.

In [ ]:
for hook in ["norm", "fanin"]:
    for metric in ["stable_rank", "rank95"]:
        plot_representation_bar(RUN_DIR, hook=hook, sampling="mixed", metric=metric, save_pdf=SAVE_PDF)

## Hidden Representation Spectrum

These plots show the singular-value decay of the hidden representation matrices. A faster decay means representation energy is concentrated in fewer directions; a flatter spectrum means high-dimensional internal features.


In [ ]:
for hook in ["norm", "fanin"]:
    plot_representation_spectrum(RUN_DIR, hook=hook, sampling="mixed", layer=5, save_pdf=SAVE_PDF)


## Fixed-time Representation Dimension

Fixed-time plots separate the geometry at specific corruption levels. Each figure is independent so it can be selected directly for a paper panel.

In [ ]:
for hook in ["norm", "fanin"]:
    for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
        plot_representation_bar(RUN_DIR, hook=hook, sampling=f"t={t:.1f}", metric="rank95", save_pdf=SAVE_PDF)

## Representation Stability under Noise Resampling

For each clean sample, we resample corruption noise several times at fixed `t`. Lower NSV means the representation is more stable to noise for the same clean point.

In [ ]:
for hook in ["norm", "fanin"]:
    for t in [0.1, 0.5, 0.9]:
        plot_stability_bar(RUN_DIR, hook=hook, t_value=t, save_pdf=SAVE_PDF)

## Sampling Quality and Sample Point-cloud Dimension

The sampler starts from Gaussian noise at high `t` and integrates back to low `t`. The 2D sample plots use the original ground-truth projection window, so generated outliers do not expand the axes and hide the manifold. The legend reports ambient high-dimensional Chamfer, precision, and recall. The 3D PCA plots fit PCA to the combined ground-truth and generated high-dimensional samples for each prediction mode, which makes off-manifold spread easier to see.


In [ ]:
for mode in ["x", "v", "eps"]:
    plot_samples(RUN_DIR, mode=mode, save_pdf=SAVE_PDF)

for mode in ["x", "v", "eps"]:
    plot_samples_pca3d(RUN_DIR, mode=mode, save_pdf=SAVE_PDF)

plot_sample_metric(RUN_DIR, metric="rank95", save_pdf=SAVE_PDF)
plot_sample_metric(RUN_DIR, metric="stable_rank", save_pdf=SAVE_PDF)


## Compact Tables

These tables are useful for a quick read before selecting figures.

In [ ]:
display(repr_df.groupby(["hook", "sampling", "mode"])[["stable_rank", "rank95"]].mean().round(2))
display(stability_df.groupby(["hook", "t", "mode"])["nsv"].mean().round(4))
display(sample_metrics_df.round(3))